# Big Data Analytics with Hadoop and Spark: Practical Applications in the Kenyan Context

##  Course Details

##### **Programme:** Master of Science in Artificial Intelligence
##### **Course:** CSA 806 - Data Mining and Big Data
##### **Module:** Module 3 - Big Data Technologies – Hadoop and Apache Spark
##### **Name:** Marrion Kiprop Cherop
##### **Reg Number:** ST62/80971/2024



# QUESTION 1 
## Spark-Based Analysis of Agricultural Production in Kenya

In [3]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, trim, upper, avg, sum as spark_sum

spark = SparkSession.builder \
    .appName("KenyaAgricultureBatchAnalysis") \
    .getOrCreate()

input_path = "../../Datasets/Kenyas_Agricultural_Production.csv"

df = spark.read.option("header", True).option("inferSchema", True).csv(input_path)

print("Original schema:")
df.printSchema()

print("Original row count:", df.count())

# Keep only useful columns and rename them
clean_df = df.select(
    col("Area").alias("area"),
    col("Item").alias("crop"),
    col("Year").alias("harvest_year"),
    col("Value").alias("value"),
    col("Unit").alias("unit"),
    col("Element").alias("element")
)

# Drop empty rows in key fields
clean_df = clean_df.dropna(subset=["area", "crop", "harvest_year", "value", "element"])

# Standardize text
clean_df = clean_df.withColumn("area", upper(trim(col("area")))) \
                   .withColumn("crop", upper(trim(col("crop")))) \
                   .withColumn("element", upper(trim(col("element")))) \
                   .withColumn("unit", upper(trim(col("unit"))))

# Keep only Kenya
clean_df = clean_df.filter(col("area") == "KENYA")

# Keep only relevant production records
clean_df = clean_df.filter(col("element").like("%PRODUCTION%"))

# Remove invalid values
clean_df = clean_df.filter(col("value") >= 0)

print("Cleaned row count:", clean_df.count())

clean_df.createOrReplaceTempView("agri")

print("\nCount by crop:")
clean_df.groupBy("crop") \
    .count() \
    .orderBy(col("count").desc()) \
    .show(20, truncate=False)

print("\nAverage production by crop:")
clean_df.groupBy("crop") \
    .agg(avg("value").alias("avg_production")) \
    .orderBy(col("avg_production").desc()) \
    .show(20, truncate=False)

print("\nProduction by year:")
spark.sql("""
    SELECT harvest_year, SUM(value) AS total_production
    FROM agri
    GROUP BY harvest_year
    ORDER BY harvest_year
""").show(truncate=False)

print("\nTop crops by total production:")
spark.sql("""
    SELECT crop, SUM(value) AS total_production
    FROM agri
    GROUP BY crop
    ORDER BY total_production DESC
""").show(20, truncate=False)

spark.stop()

Original schema:
root
 |-- Domain Code: string (nullable = true)
 |-- Domain: string (nullable = true)
 |-- Area Code (M49): integer (nullable = true)
 |-- Area: string (nullable = true)
 |-- Element Code: integer (nullable = true)
 |-- Element: string (nullable = true)
 |-- Item Code (CPC): double (nullable = true)
 |-- Item: string (nullable = true)
 |-- Year Code: integer (nullable = true)
 |-- Year: integer (nullable = true)
 |-- Unit: string (nullable = true)
 |-- Value: double (nullable = true)
 |-- Flag: string (nullable = true)
 |-- Flag Description: string (nullable = true)

Original row count: 18182


Cleaned row count: 7078

Count by crop:
+-------------------------------------------------------------------------+-----+
|crop                                                                     |count|
+-------------------------------------------------------------------------+-----+
|HEN EGGS IN SHELL, FRESH                                                 |92   |
|PYRETHRUM, DRIED FLOWERS                                                 |61   |
|CASSAVA, FRESH                                                           |61   |
|MANGOES, GUAVAS AND MANGOSTEENS                                          |61   |
|GAME MEAT, FRESH, CHILLED OR FROZEN                                      |61   |
|OTHER CITRUS FRUIT, N.E.C.                                               |61   |
|CASHEW NUTS, IN SHELL                                                    |61   |
|EDIBLE ROOTS AND TUBERS WITH HIGH STARCH OR INULIN CONTENT, N.E.C., FRESH|61   |
|OTHER STIMULANT, SPICE AND AROMATIC CROPS, N.E.C.        

This analysis was conducted using Apache Spark (PySpark) to examine agricultural production patterns in Kenya using a historical dataset containing multiple crops and livestock products across several years. The dataset was loaded into Spark as a DataFrame with schema inference enabled, and an initial inspection revealed a total of 18,182 records. After data cleaning and preprocessing, which involved removing rows with missing values, filtering out invalid entries, and selecting only production-related records for Kenya, the dataset was reduced to 7,078 valid observations . Additional transformations included renaming columns for clarity, standardizing text values, and ensuring that production values were non-negative and consistent across records. These steps ensured that the dataset was reliable and suitable for analysis.

The analysis revealed that Kenya’s agricultural production is highly diverse, encompassing staple crops, cash crops, livestock products, and horticultural outputs. Crops such as maize, cassava, beans, and potatoes represent key staples, while sugarcane and tea function as major commercial crops. Livestock products, particularly milk, also contribute significantly to overall agricultural output, indicating that Kenya’s agricultural system is both crop-based and livestock-driven. When examining average production levels, sugarcane emerged as the dominant product with an average production of approximately 3.86 million units, followed by maize at about 2.42 million units and raw milk of cattle at approximately 2.11 million units . Other crops such as potatoes, bananas, and tea also showed substantial production levels, although they were significantly lower than the leading commodities.

An analysis of production trends over time revealed a clear and consistent upward trajectory. In the early 1960s, total production was approximately 5 million units, but this steadily increased through the decades, surpassing 12 million units by the late 1970s and reaching approximately 12.86 million units by 1980 . This pattern suggests a gradual expansion of agricultural activities, likely driven by increased land utilization, improvements in farming practices, and broader agricultural development. The growth trend also reflects the increasing importance of agriculture in supporting Kenya’s population and economic needs over time.

Further analysis of total production by crop confirmed the dominance of a few key agricultural products. Sugarcane recorded the highest total production at approximately 235 million units, followed by maize at around 147 million units and raw milk of cattle at approximately 128 million units . These values are significantly higher than those of other crops, indicating a strong concentration of production in a limited number of commodities. This concentration highlights the central role of staple crops in ensuring food security, as well as the importance of certain cash crops and livestock products in supporting economic activities.

Several important insights emerge from this analysis. First, staple crops such as maize remain fundamental to Kenya’s agricultural system, reinforcing their role in national food security. Second, sugarcane stands out as a high-volume crop, likely due to its role in industrial processing and commercial agriculture. Third, livestock production, particularly milk, contributes significantly to total output, demonstrating that agriculture in Kenya is not solely crop-focused. Additionally, the steady increase in production over time reflects long-term growth in the agricultural sector. However, there is also a noticeable imbalance, where a small number of crops account for a large share of total production, while many others contribute relatively smaller amounts. In some cases, extremely high production values may also indicate differences in measurement units or aggregation levels across commodities.

This analysis demonstrates the effectiveness of Apache Spark in processing and analyzing large-scale agricultural datasets. By applying data cleaning, transformation, and aggregation techniques, it was possible to extract meaningful insights into Kenya’s agricultural production patterns. The results highlight the dominance of key crops and livestock products, the steady growth of agricultural output over time, and the structural characteristics of the sector. This approach illustrates how big data tools such as Spark can support data-driven decision-making in agriculture, particularly in understanding production trends, identifying key commodities, and informing policy and planning.

# QUESTION 2 
## Real-Time Traffic Monitoring in Nairobi Using Spark Structured Streaming

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, TimestampType
from pyspark.sql.functions import col, when, hour, avg, count

spark = SparkSession.builder \
    .appName("NairobiTrafficStreamingQuiz2") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")

schema = StructType([
    StructField("event_time", TimestampType(), True),
    StructField("junction", StringType(), True),
    StructField("vehicles_per_minute", IntegerType(), True),
    StructField("avg_speed_kmh", IntegerType(), True),
    StructField("incident_flag", IntegerType(), True)
])

input_path = "../../Datasets/sparkS"

stream_df = spark.readStream \
    .schema(schema) \
    .option("header", True) \
    .csv(input_path)

traffic_df = stream_df.withColumn(
    "congestion_level",
    when((col("avg_speed_kmh") < 15) | (col("vehicles_per_minute") > 80), "HIGH")
    .when((col("avg_speed_kmh") < 30) | (col("vehicles_per_minute") > 50), "MEDIUM")
    .otherwise("LOW")
)

alerts_df = traffic_df.filter(
    (col("congestion_level") == "HIGH") | (col("incident_flag") == 1)
)

busiest_df = traffic_df.withColumn("hour_of_day", hour(col("event_time"))) \
    .groupBy("junction", "hour_of_day") \
    .agg(
        avg("vehicles_per_minute").alias("avg_vehicles_per_minute"),
        avg("avg_speed_kmh").alias("avg_speed_kmh"),
        count("*").alias("records_received")
    ) \
    .orderBy("junction", "hour_of_day")

alert_query = alerts_df.writeStream \
    .outputMode("append") \
    .format("console") \
    .option("truncate", False) \
    .start()

busy_query = busiest_df.writeStream \
    .outputMode("complete") \
    .format("console") \
    .option("truncate", False) \
    .start()

print("Streaming has started.")
print(f"Drop CSV files into: {input_path}")
print("Press interrupt / stop cell when finished.")

spark.streams.awaitAnyTermination()

Streaming has started.
Drop CSV files into: ../../Datasets/sparkS
Press interrupt / stop cell when finished.
-------------------------------------------
Batch: 0
-------------------------------------------
+-------------------+-------------+-------------------+-------------+-------------+----------------+
|event_time         |junction     |vehicles_per_minute|avg_speed_kmh|incident_flag|congestion_level|
+-------------------+-------------+-------------------+-------------+-------------+----------------+
|2026-03-29 07:00:00|Uhuru Highway|95                 |12           |0            |HIGH            |
|2026-03-29 07:02:00|Thika Road   |110                |10           |1            |HIGH            |
|2026-03-29 08:00:00|Uhuru Highway|102                |11           |0            |HIGH            |
|2026-03-29 08:02:00|Thika Road   |120                |9            |1            |HIGH            |
|2026-03-29 08:03:00|Mombasa Road |81                 |14           |0            |HIGH

-------------------------------------------
Batch: 0
-------------------------------------------
+-------------+-----------+-----------------------+-------------+----------------+
|junction     |hour_of_day|avg_vehicles_per_minute|avg_speed_kmh|records_received|
+-------------+-----------+-----------------------+-------------+----------------+
|Jogoo Road   |7          |36.0                   |40.0         |1               |
|Jogoo Road   |8          |57.0                   |27.0         |1               |
|Jogoo Road   |17         |61.0                   |23.0         |1               |
|Jogoo Road   |21         |29.0                   |41.0         |1               |
|Mombasa Road |7          |48.0                   |34.0         |1               |
|Mombasa Road |8          |81.0                   |14.0         |1               |
|Mombasa Road |17         |90.0                   |12.0         |1               |
|Mombasa Road |21         |44.0                   |36.0         |1       

### Real-Time Traffic Monitoring in Nairobi Using Spark Structured Streaming

This task implemented a Spark Structured Streaming application to simulate real-time traffic monitoring across major Nairobi junctions. The purpose was to demonstrate how Apache Spark can process continuously arriving data, generate congestion alerts, and summarize peak traffic periods for decision-making.

The application used a file-based streaming approach in which CSV files were added into a watched input folder. Each file represented a micro-batch of traffic observations. The traffic records contained the event timestamp, junction name, vehicles per minute, average vehicle speed, and an incident flag indicating whether an abnormal event such as an accident or disruption had occurred.

The streaming application applied real-time transformations to classify congestion levels. A junction was classified as having high congestion when the average speed dropped below 15 km/h or the vehicle count exceeded 80 vehicles per minute. Medium congestion was assigned to intermediate conditions, while all other traffic states were classified as low congestion. In addition, any observation with an incident flag was treated as an alert condition regardless of the general traffic level.

The results showed clear congestion patterns. Thika Road and Uhuru Highway consistently generated real-time alerts during the morning and evening periods, reflecting their role as major traffic corridors into and within Nairobi. Mombasa Road also produced congestion alerts during the evening batch, where both vehicle density and incident occurrence contributed to traffic slowdown. Waiyaki Way and Jogoo Road showed moderate congestion during busy hours but generally lower severity than Thika Road.

The hourly aggregation results further showed that the busiest times of day occurred during the early morning commute period around 7:00 AM to 8:00 AM and again in the evening around 5:00 PM. During these periods, average traffic density was highest while average speed was lowest. Night-time observations around 9:00 PM showed much lighter traffic and higher speeds, confirming a return to normal traffic flow outside peak travel windows.

Overall, the Spark Structured Streaming application successfully demonstrated real-time traffic monitoring, congestion alerting, and time-based traffic summarization. The approach is practical for urban traffic analytics because it supports continuous monitoring and can help identify high-pressure traffic periods and high-risk junctions. In a real deployment, the same method could be connected to IoT sensors, GPS feeds, or transport APIs to support smarter traffic control and urban mobility planning in Nairobi.

# QUESTION 3 
## Livestock Population Analysis in Kenya Using Spark SQL

In [3]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, trim, upper

# -----------------------------
# 1. Start Spark Session
# -----------------------------
spark = SparkSession.builder \
    .appName("KenyaLivestockPopulation_SQL") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")

# -----------------------------
# 2. Load Dataset
# -----------------------------
input_path = "../../Datasets/Kenyas_Agricultural_Production.csv"

df = spark.read.option("header", True).option("inferSchema", True).csv(input_path)

print("Original Schema:")
df.printSchema()

print("Original Row Count:", df.count())

# -----------------------------
# 3. Clean & Transform Data
# -----------------------------
clean_df = df.select(
    col("Area").alias("area"),
    col("Item").alias("crop"),
    col("Year").alias("harvest_year"),
    col("Value").alias("value"),
    col("Unit").alias("unit"),
    col("Element").alias("element")
)

# Remove nulls
clean_df = clean_df.dropna(subset=["area", "crop", "harvest_year", "value", "unit", "element"])

# Standardize text
clean_df = clean_df.withColumn("area", upper(trim(col("area")))) \
                   .withColumn("crop", upper(trim(col("crop")))) \
                   .withColumn("unit", upper(trim(col("unit")))) \
                   .withColumn("element", upper(trim(col("element"))))

# Filter Kenya + livestock population (HEAD)
clean_df = clean_df.filter(col("area") == "KENYA")
clean_df = clean_df.filter(col("unit") == "HEAD")
clean_df = clean_df.filter(col("value") >= 0)

print("Cleaned Row Count:", clean_df.count())

clean_df.show(10, truncate=False)

# -----------------------------
# 4. Create SQL View
# -----------------------------
clean_df.createOrReplaceTempView("livestock")

# -----------------------------
# 5. Spark SQL Queries
# -----------------------------

print("\n1. Total Livestock Population by Type:")
spark.sql("""
SELECT crop AS livestock_type, SUM(value) AS total_population
FROM livestock
GROUP BY crop
ORDER BY total_population DESC
""").show(20, truncate=False)


print("\n2. Livestock Population Trend Over Time:")
spark.sql("""
SELECT harvest_year, SUM(value) AS total_population
FROM livestock
GROUP BY harvest_year
ORDER BY harvest_year
""").show(50, truncate=False)


print("\n3. Average Population by Livestock Type:")
spark.sql("""
SELECT crop, AVG(value) AS avg_population
FROM livestock
GROUP BY crop
ORDER BY avg_population DESC
""").show(20, truncate=False)


print("\n4. Top 10 Highest Livestock Population Records:")
spark.sql("""
SELECT harvest_year, crop, value
FROM livestock
ORDER BY value DESC
LIMIT 10
""").show(truncate=False)


print("\n5. Livestock Population After Year 2000:")
spark.sql("""
SELECT harvest_year, crop, value
FROM livestock
WHERE harvest_year > 2000
ORDER BY harvest_year, crop
""").show(50, truncate=False)


# -----------------------------
# 6. Stop Spark (optional)
# -----------------------------
# spark.stop()

Original Schema:
root
 |-- Domain Code: string (nullable = true)
 |-- Domain: string (nullable = true)
 |-- Area Code (M49): integer (nullable = true)
 |-- Area: string (nullable = true)
 |-- Element Code: integer (nullable = true)
 |-- Element: string (nullable = true)
 |-- Item Code (CPC): double (nullable = true)
 |-- Item: string (nullable = true)
 |-- Year Code: integer (nullable = true)
 |-- Year: integer (nullable = true)
 |-- Unit: string (nullable = true)
 |-- Value: double (nullable = true)
 |-- Flag: string (nullable = true)
 |-- Flag Description: string (nullable = true)

Original Row Count: 18182
Cleaned Row Count: 1408
+-----+------+------------+--------+----+-------+
|area |crop  |harvest_year|value   |unit|element|
+-----+------+------------+--------+----+-------+
|KENYA|CAMELS|1961        |350000.0|HEAD|STOCKS |
|KENYA|CAMELS|1962        |360000.0|HEAD|STOCKS |
|KENYA|CAMELS|1963        |380000.0|HEAD|STOCKS |
|KENYA|CAMELS|1964        |400000.0|HEAD|STOCKS |
|KENYA|CA

This task applied Apache Spark SQL to analyze livestock population patterns in Kenya using an agricultural dataset. The dataset was first loaded into Spark and cleaned by selecting relevant variables, standardizing text fields, removing null values, and filtering the data to include only records for Kenya. Unlike the previous analysis, this task focused specifically on livestock population by selecting observations where the unit of measurement was “Head,” which represents the number of animals. After preprocessing, the dataset was reduced from 18,182 records to 1,408 valid livestock-related observations . The cleaned dataset was then registered as a temporary SQL view, enabling structured querying using Spark SQL.

The analysis revealed that livestock production in Kenya is dominated by a few key animal types. Cattle, goats, and sheep recorded the highest total population levels, with cattle and goats each exceeding 780 million in total aggregated counts, followed by sheep at approximately 570 million . This indicates that these animals form the backbone of Kenya’s livestock sector, supporting food production, income generation, and rural livelihoods. Other categories such as camels and livestock by-products (e.g., milk, meat, hides, and fats) were also present but contributed relatively smaller totals compared to the primary livestock groups.

The temporal analysis of livestock population showed a gradual increase over time. In the early 1960s, total livestock population levels were approximately 27 million, but this steadily increased, reaching over 100 million by the late 2000s . This trend reflects long-term growth in livestock production, likely driven by increased demand for animal products, expansion of pastoral and mixed farming systems, and improvements in livestock management. However, some fluctuations were observed across certain years, which may be attributed to environmental factors such as drought, disease outbreaks, or changes in agricultural policies.

Further analysis of average population values confirmed that cattle and goats consistently maintain the highest average population levels, followed by sheep. Camels, while present, have significantly lower average population values, reflecting their concentration in specific ecological zones such as arid and semi-arid regions. The dataset also revealed that goats recorded the highest individual population values in specific years, with peaks exceeding 36 million animals, indicating their resilience and adaptability within Kenya’s agricultural systems.

This analysis demonstrates the effectiveness of Spark SQL in extracting meaningful insights from large-scale datasets using structured queries. By focusing specifically on livestock population, the analysis provides a deeper understanding of the composition and dynamics of Kenya’s agricultural sector. The results highlight the dominance of key livestock types, the steady growth of animal populations over time, and the importance of livestock in supporting economic and food systems. This approach shows how big data tools can support evidence-based decision-making in agriculture, particularly in monitoring livestock trends and planning resource allocation.